# EEG Signal Analysis

## What is EEG?
**Electroencephalography (EEG)** measures tiny electrical signals produced by neurons in your brain. Electrodes placed on the scalp pick up these signals. In this notebook we work with recordings from a **Muse EEG headset**, which has two relevant channels:
- **TP9** - left temporal region
- **TP10** - right temporal region

## What will we compute?

We will extract several features from the EEG signal:

| Feature | What it tells us |
|---------|------------------|
| **Band power** (δ, θ, α, β) | How much activity is in each frequency range |
| **DAR** | Delta/Alpha Ratio — higher in drowsiness/pathology |
| **DTABR** | (Delta+Theta)/(Alpha+Beta) Ratio — similar interpretation |
| **pdBSI** | How symmetric left and right brain activity is |
| **FOOOF slope** | The "background" steepness of the EEG spectrum |
| **Hjorth parameters** | Activity, mobility, and complexity of the raw signal |
| **Entropy measures** | How irregular/complex the signal is |

## Folder structure expected

```
eeg_study/
  participant_01/
    session_1/
      pre/
        closed_eyes/csv/<recording>.csv
      post/
        closed_eyes/csv/<recording>.csv
```

Each CSV file contains (at minimum) columns `TP9` and `TP10` with raw EEG amplitude values in µV, sampled at 256 Hz.

## 1. Imports and Setup

We need the following libraries:
- `numpy` / `pandas` — numerical computing and data tables
- `pywt` — PyWavelets, for the continuous wavelet transform
- `fooof` — fits the aperiodic (1/f) component of the EEG spectrum
- `matplotlib` — plotting

In [7]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import pywt
from fooof import FOOOF
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
%matplotlib inline

## 2. Configuration

### Output folders

Results (CSVs and figures) will be saved here. `os.makedirs(..., exist_ok=True)` creates the folder if it does not exist yet.

In [10]:
BASE_DIR       = 'brainhack_eeg_muse'   #TODO: root folder for all recordings
OUTPUT_FOLDER  = 'results'
FIGURES_FOLDER = 'results/figures'

os.makedirs(OUTPUT_FOLDER,  exist_ok=True)
os.makedirs(FIGURES_FOLDER, exist_ok=True)

### Signal parameters

- **FS = 256** — sampling rate in Hz. The Muse headset records 256 samples per second.
- **WAVENUMBER = 6** — a parameter of the Morlet wavelet. Higher values give better frequency resolution but worse time resolution (a trade-off called the *time-frequency uncertainty principle*).
- **FREQS** — the set of frequencies (in Hz) at which we will estimate power. We use 0.5–31 Hz in steps of 0.1 Hz.

In [13]:
FS         = 256                        # sampling rate (Hz)
WAVENUMBER = 6                          # Morlet wavelet parameter
FREQS      = np.arange(0.5,31,0.1) #TODO: frequencies to analyse (Hz)

### EEG frequency bands

Brain activity is traditionally divided into frequency bands, each associated with different mental states:

| Band | Range | Typical association |
|------|-------|---------------------|
| **Delta** | 1–3 Hz | Deep sleep, anaesthesia |
| **Theta** | 4–7 Hz | Drowsiness, meditation, memory encoding |
| **Alpha** | 8–13 Hz | Relaxed wakefulness, eyes closed |
| **Beta** | 14–20 Hz | Active thinking, alertness |

> **Try it:** What band do you expect to be strongest when eyes are closed and the participant is resting? Why?

In [15]:
#TODO: put the frequency bands of interest here (Hz)
BANDS = {
    'delta': (1,4),
    'theta': (4,8),
    'alpha': (8,14),
    'beta':  (14,20),
}

### FOOOF settings

The EEG power spectrum has two components:
1. **Aperiodic (1/f) component** — power decreases steadily with frequency (like 1/f). This is the "background" noise floor.
2. **Periodic (oscillatory) component** — peaks sitting on top of the aperiodic component (e.g. the alpha peak at ~10 Hz).

**FOOOF** (Fitting Oscillations & One-Over-F) separates these two components. The *aperiodic slope* (steepness of the 1/f drop) changes with age, arousal, and pathology.

Key settings:
- `peak_width_limits` — minimum and maximum width (Hz) of a peak to be detected
- `max_n_peaks` — how many peaks to allow
- `peak_threshold` — how many standard deviations above the aperiodic fit a peak must be to count
- `aperiodic_mode='fixed'` — models aperiodic component as a straight line in log-log space

In [17]:
FOOOF_RANGE    = [0.5, 30]   # frequency range to fit (Hz)
FOOOF_SETTINGS = dict(
    peak_width_limits=[0.5, 12],  # min/max peak width in Hz
    max_n_peaks=10,            #TODO: allow any number of peaks
    min_peak_height=0.0,
    peak_threshold=2.0,            # peaks must be >2 SD above aperiodic
    aperiodic_mode='fixed',        # straight-line aperiodic fit
    verbose=False,
)

### Electrode names and optional filters

`LEFT_ELEC` and `RIGHT_ELEC` must match the column names in your CSV files.

The `FILTER_*` variables let you select a subset of recordings. Set them to `None` to include everything.

**Example:** to analyse only participant 1 with eyes open:
```python
FILTER_PARTICIPANTS = [1]
FILTER_CONDITIONS   = ['open_eyes']
```

In [18]:
#TODO: specify the electrode names
LEFT_ELEC  = ''
RIGHT_ELEC = ''

# Set to None to include all, or provide a list to filter
FILTER_PARTICIPANTS = [1]   # e.g. [1, 3, 5]
FILTER_SESSIONS     = [1]   # e.g. [1]
FILTER_TIMINGS      = ['pre']   # e.g. ['pre', 'post']
FILTER_CONDITIONS   = ['closed_eyes']   # e.g. ['open_eyes']

# Colors for each frequency band (used in plots)
BAND_REGIONS = [
    ('Delta\n1-3 Hz',  1,  3,  '#a8d5e2'),
    ('Theta\n4-7 Hz',  4,  7,  '#f6d365'),
    ('Alpha\n8-13 Hz', 8,  13, '#b5ead7'),
    ('Beta\n14-20 Hz', 14, 20, '#f7a8b8'),
]

## 3. Loading Data

### Discovering recording files

The function `discover_recordings()` walks the `eeg_study/` folder tree and returns a list of dictionaries — one entry per recording file found.

Each dictionary contains:
- `csv_path` — path to the CSV file
- `participant`, `session` — numbers extracted from folder names
- `timing` — `'pre'` or `'post'` (before/after intervention)
- `condition` — `'open_eyes'` or `'closed_eyes'`
- `label` — a human-readable identifier like `P01_S1_pre_open_eyes`

The `FILTER_*` variables defined above are applied here.

In [20]:
# Path helpers
def condition_path(p, s, timing, condition):
    return os.path.join(
        BASE_DIR,
        f"participant_{p:02d}",
        f"session_{s}",
        timing,
        condition,
    )

In [22]:
def discover_recordings(base_dir=BASE_DIR, participants_csv='participants.csv'):
    # Load group assignments from participants.csv
    group_map = {}
    if os.path.isfile(participants_csv):
        df_p = pd.read_csv() #TODO: load participants.csv
        df_p.columns = df_p.columns.str.strip().str.lower()
        if 'group' in df_p.columns:
            df_p['id'] = pd.to_numeric(df_p['id'], errors='coerce').astype('Int64')
            group_map = dict(zip(df_p['id'].astype(int), df_p['group'].str.strip()))
        else:
            print("  [!] 'group' column not found in participants.csv — group will be None.")
    else:
        print(f"  [!] '{participants_csv}' not found — group will be None.")

    records = []
    p_dirs = sorted(glob.glob(os.path.join(base_dir, 'participant_*')))

    for p_dir in p_dirs:
        p_name = os.path.basename(p_dir)
        try:
            p_num = int(p_name.split('_')[1])
        except (IndexError, ValueError):
            continue

        if FILTER_PARTICIPANTS and p_num not in FILTER_PARTICIPANTS:
            continue

        for s_dir in sorted(glob.glob(os.path.join(p_dir, 'session_*'))):
            try:
                s_num = int(os.path.basename(s_dir).split('_')[1])
            except (IndexError, ValueError):
                continue

            if FILTER_SESSIONS and s_num not in FILTER_SESSIONS:
                continue

            for timing in ('pre', 'post'):
                if FILTER_TIMINGS and timing not in FILTER_TIMINGS:
                    continue

                for condition in ('open_eyes', 'closed_eyes'):
                    if FILTER_CONDITIONS and condition not in FILTER_CONDITIONS:
                        continue

                    csv_dir = construct_path(p_num, s_num, timing, condition) #TODO: construct the path to the CSV file for this participant/session/timing/condition
                    if not os.path.isdir(csv_dir):
                        continue

                    csvs = sorted(
                        glob.glob(os.path.join(csv_dir, '*.csv')),
                        reverse=True,
                    )
                    if not csvs:
                        continue

                    if len(csvs) > 1:
                        print(f"  [!] Multiple CSVs in {csv_dir} — using {os.path.basename(csvs[0])}")

                    label = f"P{p:02d}_S{s:02d}_{timing}_{condition}" #TODO: create a label for this recording (e.g. 'P01_S01_pre_open_eyes')
                    records.append({
                        'csv_path':    csvs[0],
                        'participant': p_num,
                        'session':     s_num,
                        'timing':      timing,
                        'condition':   condition,
                        'label':       label,
                        'group':       group_map.get(p_num, None),  # ← NEW
                    })

    return records

## 4. Computing the Power Spectrum

### What is a Power Spectral Density (PSD)?

The PSD tells us **how much signal power exists at each frequency**. A typical EEG PSD has:
- High power at low frequencies (1–4 Hz)
- A visible *alpha peak* around 8–13 Hz when eyes are closed
- Power that generally decreases at higher frequencies (the 1/f shape)

### Continuous Wavelet Transform (CWT) with a Morlet wavelet

We compute the PSD using a **Morlet wavelet** instead of a simple Fourier transform. Why?
- The wavelet transform gives us **time-resolved power** — we can see how power changes over time.
- A Morlet wavelet is a sinusoidal wave multiplied by a Gaussian window, which gives a good balance between time and frequency resolution.

The function returns a 2D array of shape `(n_freqs, n_samples)` — power at each frequency and each time point.

After averaging across time (`mean(axis=1)`), we get the mean PSD — a 1D array of shape `(n_freqs,)`.

$$\frac{wavenumber*fs}{2*\pi*freqs}
$$

**References:**
- https://pywavelets.readthedocs.io/en/latest/ref/cwt.html

In [24]:
def morlet_power(signal, fs=FS, freqs=FREQS, wavenumber=WAVENUMBER):
    """
    Compute instantaneous power at each frequency using a Morlet wavelet.

    Parameters
    ----------
    signal     : 1D numpy array of EEG voltage values (µV)
    fs         : sampling rate in Hz
    freqs      : array of frequencies to analyse (Hz)
    wavenumber : controls time vs frequency resolution

    Returns
    -------
    power : 2D array of shape (n_freqs, n_samples) — power in µV²
    """
    # Convert frequencies to wavelet 'widths' (number of cycles)
    # Higher frequency → narrower wavelet in time
    widths = (wavenumber * fs) / (2*np.pi*freqs) #TODO: convert freqs to widths using the wavenumber parameter 

    # Apply the continuous wavelet transform
    # 'cmor1.5-1.0' is the complex Morlet wavelet
    coeffs, _ = pywt.cwt(signal, widths, wavelet='cmor1.5-1.0', sampling_period=1/fs)

    #TODO: Power = |complex amplitude|²
    power = np.abs(coeffs) ** 2 # calling np.abs so that we don't double the length of a list
    return 

### Extracting power within a frequency band

Once we have the mean PSD (average over time), we can sum the power within a frequency band — for example, sum all values between 8 and 13 Hz to get the alpha power.

`band_power()` creates a boolean *mask* to select only the frequencies within the band, then sums those values.

In [26]:
def band_power(psd, freqs=FREQS, band=None):
    """
    Sum the power within a given frequency band.

    Parameters
    ----------
    psd   : 1D array — mean power spectrum (µV²/Hz)
    freqs : 1D array — frequency values matching psd
    band  : tuple (f_low, f_high) in Hz

    Returns
    -------
    float — total power within the band
    """
    #TODO: Boolean mask: True for frequencies inside the band
    mask = (freqs >= band[0] & freqs < band[1])
    return float(np.sum(psd[mask]))

## 5. EEG Metrics

### Brain Symmetry Index (pdBSI)

The **power-difference Brain Symmetry Index** measures how asymmetric brain activity is between the left and right hemispheres.

**Formula:**
$$\text{pdBSI} = \text{mean}\left(\frac{|P_R(f) - P_L(f)|}{P_R(f) + P_L(f)}\right)$$

where the mean is taken over all frequencies in the analysis range.

- **pdBSI = 0** → perfectly symmetric (left and right power are equal at every frequency)
- **pdBSI = 1** → maximum asymmetry (all power is on one side)

> **Question:** Would you expect a higher or lower pdBSI during meditation compared to rest? Why might the brain be more or less symmetric?

In [27]:
def compute_pdBSI(left_psd, right_psd, freqs=FREQS, f_low=1.0, f_high=30.0):
    """
    Power-difference Brain Symmetry Index.
    Returns a value between 0 (symmetric) and 1 (maximally asymmetric).
    """
    # Select the analysis frequency range
    mask = (freqs >= f_low) & (freqs <= f_high)
    L, R = left_psd[mask], right_psd[mask]

    #TODO: Compute |R - L| / (R + L) at each frequency, avoiding division by zero
    numer = np.abs(R - L)
    denom = R + L
    ratio = np.divide(numer, denom, out=np.zeros_like(numer), where=denom!=0)
      # we postulate 0 (balance) where the total power is zero
    ## we could also use NaNs there:
    # ratio = np.divide(numer, denom, out=np.full_like(numer, np.nan, dtype=float), where=denom!=0)
    
    # Average across frequencies
    return float(np.mean(ratio))

### Delta/Alpha Ratio (DAR) and (Delta+Theta)/(Alpha+Beta) Ratio (DTABR)

These ratios compare **slow** brain activity (delta, theta) to **fast** activity (alpha, beta).

| Ratio | Formula | High value means |
|-------|---------|------------------|
| **DAR** | δ / α | More slow activity relative to alpha |
| **DTABR** | (δ+θ) / (α+β) | More slow activity overall |

**When are these ratios high?**
- Drowsiness or sleep
- Deep relaxation
- Pathological states (e.g. brain injury, some neurological disorders)

**When are these ratios low?**
- Alert, eyes-open wakefulness
- Strong alpha oscillations (relaxed, eyes closed)

In [ ]:
def compute_dar(psd, freqs=FREQS):
    """Delta / Alpha Ratio — higher values indicate more slow-wave activity."""
    d = band_power(psd, freqs, BANDS['delta'])
    a = band_power(psd, freqs, BANDS['alpha'])
    #TODO: Compute the ratio, handling division by zero
    return 


def compute_dtabr(psd, freqs=FREQS):
    """(Delta + Theta) / (Alpha + Beta) Ratio."""
    d  = band_power(psd, freqs, BANDS['delta'])
    th = band_power(psd, freqs, BANDS['theta'])
    a  = band_power(psd, freqs, BANDS['alpha'])
    b  = band_power(psd, freqs, BANDS['beta'])
    #TODO: Compute the ratio, handling division by zero
    return 

### FOOOF — Fitting Oscillations & One-Over-F

The EEG power spectrum follows a **1/f** (or "pink noise") distribution: power decreases as frequency increases, roughly as `Power ∝ 1 / frequency^slope`.

FOOOF separates the spectrum into:
1. **Aperiodic component** — the 1/f background. Described by: `log(Power) = intercept − slope × log(frequency)`.
2. **Periodic component** — Gaussian peaks sitting on top (e.g. the alpha peak).

The **aperiodic slope** is clinically and cognitively relevant:
- Steeper slope → more 1/f character, associated with less E/I balance disruption
- Changes with age, arousal, cognitive load, and various brain disorders

> **Visualisation note:** After running the analysis, call `plot_fooof_fit(pid)` to see the fitted model overlaid on the raw PSD.

In [ ]:
def fit_fooof(psd, freqs=FREQS):
    """
    Fit the FOOOF model to extract the aperiodic slope and intercept.

    Returns
    -------
    slope     : float — steepness of the 1/f drop (positive = slopes down)
    intercept : float — overall power level (log scale)
    Returns (nan, nan) if fitting fails.
    """
    fm = FOOOF(**FOOOF_SETTINGS)
    try:
        fm.fit(freqs, psd, FOOOF_RANGE)
        # aperiodic_params_ = [intercept, slope]
        #TODO: return slope and intercept in the correct order
        return 
    except Exception:
        return np.nan, np.nan

### Relative Band Power

**Absolute** band power (µV²) depends on the overall signal amplitude, which varies between participants and sessions.

**Relative** band power normalises each band by the total power across all bands combined:

$$\text{rel\_alpha} = \frac{P_\alpha}{P_\delta + P_\theta + P_\alpha + P_\beta}$$

This makes it easier to compare across participants, because it reflects the *proportion* of activity in each band rather than the raw amplitude.

In [ ]:
def compute_relative_power(psd, freqs=FREQS):
    """
    Compute relative band power for each band.

    Returns a dict with keys 'rel_delta', 'rel_theta', 'rel_alpha', 'rel_beta'.
    Each value is the fraction of total band power in that band (sums to 1.0).
    """
    # Compute absolute power for each band
    band_powers = {name: band_power(psd, freqs, rng)
                   for name, rng in BANDS.items()}

    total = sum(band_powers.values())
    if total == 0:
        return {f'rel_{name}': np.nan for name in BANDS}

    # Divide each band by the total to get proportions
    #TODO: return the relative power for each band
    return 

## 6. Signal Complexity Measures

### Hjorth Parameters

Hjorth parameters are simple **time-domain** measures of signal properties, computed directly from the raw EEG signal (no frequency transform needed):

| Parameter | Formula | Meaning |
|-----------|---------|----------|
| **Activity** | Var(x) | Signal power / amplitude variance |
| **Mobility** | √(Var(x') / Var(x)) | Proportion of standard deviation of power spectrum (mean frequency proxy) |
| **Complexity** | Mobility(x') / Mobility(x) | How much the shape of the spectrum deviates from a pure sine wave |

where x' is the first derivative (difference between consecutive samples) and x'' is the second derivative.

**Intuition:**
- High **activity** = large amplitude oscillations
- High **mobility** = faster oscillations (higher mean frequency)
- High **complexity** = signal that is more complex (more irregular, less like a single tone)

In [ ]:
def compute_hjorth(signal):
    """
    Compute Hjorth parameters: Activity, Mobility, Complexity.

    Parameters
    ----------
    signal : 1D numpy array of raw EEG values

    Returns
    -------
    activity   : float — variance of the signal
    mobility   : float — std of 1st derivative / std of signal
    complexity : float — mobility of 1st derivative / mobility of signal
    """
    diff1 = np.diff(signal)     # first derivative  (x')
    diff2 =      #TODO: second derivative (x'')

    var0 =       #TODO: variance of original signal
    var1 =       #TODO: variance of first derivative
    var2 =       #TODO: variance of second derivative

    activity = var0

    #TODO: Mobility: undefined if variance is zero
    mobility = 

    #TODO: Complexity: ratio of mobilities (signal vs its derivative)
    complexity = 

    return activity, mobility, complexity

### Entropy Measures

**Entropy** quantifies the *unpredictability* or *complexity* of a signal. In EEG:
- **Low entropy** → regular, predictable signal (e.g. a strong alpha oscillation, or anaesthesia)
- **High entropy** → irregular, complex signal (e.g. normal awake activity)

We compute three different entropy measures:

#### 1. Spectral Entropy
Treats the normalised PSD as a probability distribution and computes its **Shannon entropy**:
$$H_{spectral} = -\sum_f p(f) \log_2 p(f)$$

#### 2. Sample Entropy (SampEn)
Measures how often a short pattern in the signal recurs. If a pattern of length `m` recurs, does it also recur when extended to length `m+1`?

- Low SampEn → patterns often repeat (regular/predictable signal)
- High SampEn → patterns rarely repeat (complex/irregular signal)

> Note: Sample entropy is slow to compute (O(N²)) — the code downsamples long signals first.

#### 3. Permutation Entropy
Replaces signal values with **ordinal patterns** (the rank ordering of short windows) and computes the entropy of these patterns. It is much faster than Sample Entropy and robust to noise.

> **Question:** During eyes-closed rest with a strong alpha rhythm, do you expect entropy to be higher or lower compared to eyes-open? Why?

In [ ]:
def compute_spectral_entropy(psd, freqs=FREQS, f_low=0.5, f_high=30.0):
    """
    Shannon entropy of the normalised power spectrum.

    Higher = more uniformly distributed power (complex/broadband signal).
    Lower  = power concentrated in a narrow band (e.g. strong alpha peak).
    """
    # Select the frequency range of interest
    mask = (freqs >= f_low) & (freqs <= f_high)
    seg  = psd[mask]

    #TODO: Normalise to make it a probability distribution (sums to 1)
    seg = 

    # Replace zeros to avoid log(0) = -inf
    seg = np.where(seg > 0, seg, 1e-12)

    # Shannon entropy formula: H = -sum(p * log2(p))
    return float(-np.sum(seg * np.log2(seg)))

In [ ]:
def compute_sample_entropy(signal, m=2, r_factor=0.2):
    """
    Sample Entropy (SampEn): measures signal regularity.

    Parameters
    ----------
    m        : template length (2 is standard for EEG)
    r_factor : tolerance = r_factor * std(signal)
               Typical range: 0.1–0.25

    Lower SampEn = more regular / predictable.
    Higher SampEn = more complex / irregular.
    """
    # Downsample very long signals to keep computation tractable
    if len(signal) > 2048:
        step = len(signal) // 2048
        signal = signal[::step]

    n = len(signal)
    r = r_factor * np.std(signal)   # tolerance threshold

    def _count_matches(templates, length):
        """Count pairs of templates closer than r (Chebyshev distance)."""
        count = 0
        for i in range(len(templates)):
            for j in range(i + 1, len(templates)):
                if np.max(np.abs(templates[i] - templates[j])) < r:
                    count += 1
        return count

    # Build all length-m and length-(m+1) templates
    templates_m  = np.array([signal[i:i + m]     for i in range(n - m)])
    templates_m1 = np.array([signal[i:i + m + 1] for i in range(n - m)])

    B = _count_matches(templates_m,  m)      # matches of length m
    A = _count_matches(templates_m1, m + 1)  # matches of length m+1

    if B == 0:
        return np.nan   # undefined if no matches found

    #TODO: SampEn = -log(A/B)
    return 

In [ ]:
def compute_permutation_entropy(signal, order=3, delay=1, normalize=True):
    """
    Permutation Entropy: fast ordinal complexity measure.

    Parameters
    ----------
    order    : number of samples per ordinal pattern (3–7 typical)
    delay    : time steps between samples in each pattern
    normalize: if True, result is in [0, 1]

    Lower = more regular.  Higher = more complex / random.
    """
    from itertools import permutations as _perms
    from math import factorial, log2

    n = len(signal)
    all_perms = list(_perms(range(order)))
    perm_idx  = {p: i for i, p in enumerate(all_perms)}

    # Count how often each ordinal pattern occurs
    counts = np.zeros(factorial(order))
    for i in range(n - (order - 1) * delay):
        # Get the rank ordering of this window
        pattern = tuple(np.argsort(signal[i:i + order * delay:delay]))
        counts[perm_idx[pattern]] += 1

    #TODO: Remove zero counts, compute probabilities
    counts = 
    #TODO: Shannon entropy formula: H = -sum(p * log2(p))
    p  = counts / counts.sum()
    pe = 
    
    if normalize:
        pe /= log2(factorial(order))   # divide by max possible entropy

    return float(pe)

## 7. Main Processing Pipeline

### The `process_file` function

This function takes a single recording descriptor (from `discover_recordings`) and:
1. Loads the CSV file
2. Runs the Morlet wavelet transform on both TP9 and TP10
3. Computes all metrics (band power, pdBSI, DAR, DTABR, FOOOF, Hjorth, entropy)
4. Saves a per-recording PSD CSV file
5. Returns a flat dictionary of all metrics

The results are stored internally as lists and later assembled into a pandas DataFrame.

In [18]:
def process_file(rec):
    """
    Process one recording and return a dictionary of all computed metrics.

    Parameters
    ----------
    rec : dict from discover_recordings(), with keys csv_path, label, etc.

    Returns
    -------
    row : flat dict of metrics, ready to be added to a DataFrame
    """
    label = rec['label']

    # --- Load the CSV file ---
    df   = pd.read_csv(rec['csv_path']).dropna(subset=[LEFT_ELEC, RIGHT_ELEC])
    tp9  = df[LEFT_ELEC].to_numpy(float)   # left electrode signal
    tp10 = df[RIGHT_ELEC].to_numpy(float)  # right electrode signal

    # --- Compute power spectra using Morlet wavelets ---
    # pwr9 / pwr10 : shape (n_freqs, n_samples) — power at each frequency and time
    pwr9  = morlet_power(tp9)
    pwr10 = morlet_power(tp10)

    # Average over time to get mean PSD: shape (n_freqs,)
    psd9  = pwr9.mean(axis=1)
    psd10 = pwr10.mean(axis=1)

    # Average of both electrodes
    psd_avg = (psd9 + psd10) / 2

    # --- Fit FOOOF to get aperiodic slope ---
    slope9,  ic9  = fit_fooof(psd9)
    slope10, ic10 = fit_fooof(psd10)

    # --- Hjorth parameters on raw signals ---
    act9,  mob9,  comp9  = compute_hjorth(tp9)
    act10, mob10, comp10 = compute_hjorth(tp10)

    # --- Relative band power ---
    rel9    = compute_relative_power(psd9)
    rel10   = compute_relative_power(psd10)
    rel_avg = compute_relative_power(psd_avg)

    # --- Assemble the results row ---
    row = {
        # Identity / metadata columns
        'participant_id': label,
        'participant':    rec['participant'],
        'session':        rec['session'],
        'timing':         rec['timing'],
        'condition':      rec['condition'],
        'group':          rec['group'],  

        'n_samples':  len(tp9),
        'duration_s': len(tp9) / FS,

        # Symmetry and band ratios
        'pdBSI':      compute_pdBSI(psd9, psd10),
        'DAR':        compute_dar(psd_avg),
        'DTABR':      compute_dtabr(psd_avg),
        'DAR_TP9':    compute_dar(psd9),
        'DAR_TP10':   compute_dar(psd10),
        'DTABR_TP9':  compute_dtabr(psd9),
        'DTABR_TP10': compute_dtabr(psd10),

        # FOOOF aperiodic parameters
        'fooof_slope_tp9':      slope9,
        'fooof_intercept_tp9':  ic9,
        'fooof_slope_tp10':     slope10,
        'fooof_intercept_tp10': ic10,

        # Hjorth parameters
        'hjorth_activity_tp9':    act9,
        'hjorth_mobility_tp9':    mob9,
        'hjorth_complexity_tp9':  comp9,
        'hjorth_activity_tp10':   act10,
        'hjorth_mobility_tp10':   mob10,
        'hjorth_complexity_tp10': comp10,

        # Entropy measures
        'spectral_entropy_tp9':     compute_spectral_entropy(psd9),
        'spectral_entropy_tp10':    compute_spectral_entropy(psd10),
        'sample_entropy_tp9':       compute_sample_entropy(tp9),
        'sample_entropy_tp10':      compute_sample_entropy(tp10),
        'permutation_entropy_tp9':  compute_permutation_entropy(tp9),
        'permutation_entropy_tp10': compute_permutation_entropy(tp10),

        # Internal — stripped before saving to CSV (stored in psd_store)
        '_psd9':  psd9,
        '_psd10': psd10,
        '_pwr9':  pwr9,
        '_pwr10': pwr10,
    }

    # --- Absolute band power for each band and electrode ---
    for name, band in BANDS.items():
        row[f'{name}_tp9']  = band_power(psd9,    band=band)
        row[f'{name}_tp10'] = band_power(psd10,   band=band)
        row[f'{name}_avg']  = band_power(psd_avg, band=band)

    # --- Relative band power ---
    for name in BANDS:
        row[f'rel_{name}_tp9']  = rel9[f'rel_{name}']
        row[f'rel_{name}_tp10'] = rel10[f'rel_{name}']
        row[f'rel_{name}_avg']  = rel_avg[f'rel_{name}']

    # --- Save per-recording PSD to CSV ---
    pd.DataFrame({
        'frequency':  FREQS,
        'power_tp9':  psd9,
        'power_tp10': psd10,
        'power_avg':  psd_avg,
    }).to_csv(os.path.join(OUTPUT_FOLDER, f'{label}_mean_psd.csv'), index=False)

    return row

## 8. Running the Analysis

The cell below:
1. Calls `discover_recordings()` to find all CSV files
2. Loops over each recording and calls `process_file()`
3. Collects all results into `raw_results`

> **Note:** The wavelet transform is computationally intensive. Each recording takes approximately 45 seconds. Be patient!

In [ ]:
# Discover all recordings that match the current filter settings
recordings = discover_recordings()
print(f'Found {len(recordings)} recording(s).\n')

# Load already-computed results if the file exists
results_path = os.path.join(OUTPUT_FOLDER, 'per_recording_metrics.csv')
if os.path.isfile(results_path):
    df_existing = pd.read_csv(results_path)
    done_labels = set(df_existing['participant_id'].tolist())
    print(f'Found {len(done_labels)} already-processed recording(s) — will skip them.')
else:
    df_existing = pd.DataFrame()
    done_labels = set()

# Print a summary of what was found
for r in recordings:
    status = '  [skip]' if r['label'] in done_labels else '  [run]  '
    print(f"{status}  {r['label']:<40}  {r['csv_path']}")

# Process only new recordings
raw_results = []
for rec in recordings:
    #TODO: skip if rec['label'] is in done_labels
    #TODO: append the result of process_file(rec) to raw_results

### Saving results to CSV

We separate the large power arrays (`_psd9`, `_psd10`, `_pwr9`, `_pwr10`) from the scalar metrics:
- Scalar metrics → saved as `results/per_recording_metrics.csv`
- Power arrays → kept in memory in `psd_store` (used for plotting)

The `psd_store` dictionary maps each `participant_id` label to its power data.

In [ ]:
psd_store = {}
clean_results = []

for r in raw_results:
    psd_store[r['participant_id']] = {
        'psd9':  r.pop('_psd9'),
        'psd10': r.pop('_psd10'),
        'pwr9':  r.pop('_pwr9'),
        'pwr10': r.pop('_pwr10'),
    }
    clean_results.append(r)

# Merge new results with existing ones
if clean_results:
    df_new = pd.DataFrame(clean_results)
    #TODO: concatenate df_existing and df_new, handling the case where df_existing may be empty
    df_results = 
    df_results.to_csv(results_path, index=False)
    print(f'\nAdded {len(clean_results)} new recording(s). Total: {len(df_results)}.')
else:
    df_results = df_existing
    print('\nNo new recordings to process — loaded existing results.')

# Reload psd_store for any recordings that were skipped (loaded from CSV)
# These were saved by process_file as individual per-recording PSD files
missing_pids = [pid for pid in df_results['participant_id'] if pid not in psd_store]
if missing_pids:
    print(f'\nReloading PSD arrays for {len(missing_pids)} skipped recording(s)...')
    for pid in missing_pids:
        psd_path = os.path.join(OUTPUT_FOLDER, f'{pid}_mean_psd.csv')
        if os.path.isfile(psd_path):
            #TODO: load the PSD from the CSV file and store it in psd_store[pid]
            psd_df = 
            psd_store[pid] = {
                'psd9':  psd_df['power_tp9'].to_numpy(),
                'psd10': psd_df['power_tp10'].to_numpy(),
                # pwr (full 2D time-resolved power) is not saved to disk —
                # spectrograms will be unavailable for skipped recordings
                'pwr9':  None,
                'pwr10': None,
            }
        else:
            print(f'  [!] PSD file not found for {pid} — skipping.')
    print('  Done.')

## 9. Results Overview

The styled table below shows the key metrics for each recording.

**How to read it:**
- `pdBSI` — higher = more asymmetric (red = more asymmetric)
- `DAR` / `DTABR` — higher = more slow activity (red = more slow activity)
- `fooof_slope` — steepness of the 1/f background spectrum

In [21]:
# Sort by participant, session, timing, condition for readability
df_results.sort_values(['participant', 'session', 'timing', 'condition'])[
    ['participant_id', 'duration_s', 'pdBSI', 'DAR', 'DTABR',
     'fooof_slope_tp9', 'fooof_slope_tp10']
].style.background_gradient(subset=['pdBSI', 'DAR', 'DTABR'], cmap='YlOrRd')\
        .format(precision=3)

In [22]:
# Group by session / timing / condition and show means
df_results.groupby(['session', 'timing', 'condition'])[
    ['pdBSI', 'DAR', 'DTABR', 'fooof_slope_tp9', 'fooof_slope_tp10']
].mean().style.format(precision=3)

In [23]:
if 'group' in df_results.columns:
    display(
        df_results.groupby(['group', 'timing', 'condition'])[
            ['pdBSI', 'DAR', 'DTABR', 'fooof_slope_tp9', 'fooof_slope_tp10']
        ].mean().style.format(precision=3)
          .set_caption('Mean metrics by group × timing × condition')
    )

## 10. Visualisation

### Helper functions

`_add_band_vspans()` adds coloured background regions to a plot to highlight the frequency bands — making it easy to see which features correspond to which band.

`_short()` creates a compact label from the long participant ID string, e.g. `P01_S1_pre_open_eyes` → `P01 S1 pre OE`.

In [ ]:
def _add_band_vspans(ax, vertical=True):
    """Add coloured background regions for each frequency band to a plot axis."""
    for label, f_lo, f_hi, color in BAND_REGIONS:
        if vertical:
            ax.axvspan(f_lo, f_hi, alpha=0.18, color=color, lw=0)
        else:
            ax.axhspan(f_lo, f_hi, alpha=0.15, color=color, lw=0)


def _short(pid):
    """
    Return a compact label: 'P01_S1_pre_open_eyes' → 'P01 S1 pre OE'
    """
    parts = pid.split('_')
    cond_abbr = 'OE' if 'open' in parts else 'CE'
    #TODO: Return participant session timing condition'P01 S1 pre OE'
    return 

### Spectrogram

A **spectrogram** shows how power at each frequency changes over time — it is the full 2D output of the wavelet transform, plotted as a colour map.

- **x-axis** — time (seconds)
- **y-axis** — frequency (Hz)
- **colour** — log₁₀ power (brighter = more power)

Look for:
- Horizontal bands of high power → sustained oscillations (e.g. alpha at ~10 Hz, eyes closed)
- Vertical stripes → transient artefacts (e.g. movement)
- Changes before/after an intervention

In [ ]:
def plot_spectrogram(pid, electrode='TP9', save=False):
    key = 'pwr9' if electrode == 'TP9' else 'pwr10'
    elec_col = LEFT_ELEC if electrode == 'TP9' else RIGHT_ELEC
    power = psd_store[pid][key]

    if power is None:
        rec = next(
            (r for r in discover_recordings() if r['label'] == pid),
            None
        )
        if rec is None:
            print(f'  [!] Nie znaleziono pliku CSV dla {pid} — pomijam.')
            return
        print(f'  [recalc] Przeliczam spektrogram dla {pid} {electrode} z CSV...')
        df_sig = pd.read_csv(rec['csv_path']).dropna(subset=[elec_col])
        signal = df_sig[elec_col].to_numpy(float)
        power  = morlet_power(signal)
        psd_store[pid][key] = power

    times = np.arange(power.shape[1]) / FS

    fig, ax = plt.subplots(figsize=(12, 4))
    pcm = ax.pcolormesh(times, FREQS, np.log10(power + 1e-12),
                        shading='auto', cmap='inferno')
    fig.colorbar(pcm, ax=ax, pad=0.01, label='log10 Power (µV²/Hz)')

    _add_band_vspans(ax, vertical=False)
    for label, f_lo, f_hi, _ in BAND_REGIONS:
        ax.text(times[-1]*1.01, (f_lo + f_hi)/2, label,
                va='center', ha='left', fontsize=7, color='dimgray')

    ax.set(xlabel='Time (s)', ylabel='Frequency (Hz)',
           title=f'Spectrogram — {_short(pid)} | {electrode}',
           ylim=(FREQS[0], FREQS[-1]))
    fig.tight_layout()
    if save:
        fig.savefig(os.path.join(FIGURES_FOLDER, f'{pid}_{electrode}_spectrogram.png'), dpi=150)
    plt.show()

### Mean Power Spectrum

Averaging the spectrogram over time gives the **mean PSD** — the typical frequency profile for the whole recording.

On a **log y-axis** (semilogy), the 1/f background appears as a straight line with a negative slope. Peaks above this line are the oscillatory components.

The **alpha peak** (8–13 Hz) is often visible as a bump in closed-eyes recordings.

In [26]:
def plot_mean_psd(pid, save=False):
    """Plot the mean power spectrum for both electrodes."""
    psd9  = psd_store[pid]['psd9']
    psd10 = psd_store[pid]['psd10']

    fig, ax = plt.subplots(figsize=(9, 4))
    _add_band_vspans(ax)   # coloured band regions

    # semilogy = log scale on y-axis
    ax.semilogy(FREQS, psd9,  color='#2d6a9f', lw=1.8, label='TP9 (left)')
    ax.semilogy(FREQS, psd10, color='#c0392b', lw=1.8, label='TP10 (right)', ls='--')

    ax.set(xlabel='Frequency (Hz)', ylabel='Power (µV²/Hz, log)',
           title=f'Mean Power Spectrum — {_short(pid)}',
           xlim=(FREQS[0], FREQS[-1]))
    ax.legend(fontsize=9)
    fig.tight_layout()
    if save:
        fig.savefig(os.path.join(FIGURES_FOLDER, f'{pid}_mean_psd.png'), dpi=150)
    plt.show()

### Band Ratio Bar Chart

This plot displays DAR and DTABR side-by-side for all recordings.

- The **dashed line at y=1** marks the boundary between slow-dominant (>1) and fast-dominant (<1) activity.
- Compare open-eyes vs closed-eyes recordings — which has higher DAR? Is this expected?

In [ ]:
#TODO: Make plot to show band ratio
def plot_band_ratios(subset=None, save=False):


### Brain Symmetry Index Plot

Horizontal bar chart coloured by pdBSI value (green = symmetric, red = asymmetric).

In [ ]:
#TODO: Make plot to show pdBSI
def plot_pdBSI(subset=None, save=False):

### FOOOF Fit Visualisation

This plot shows the FOOOF model overlaid on the raw PSD:
- **Grey** — raw PSD (full range)
- **Dark grey** — raw PSD within the fit range
- **Orange dashed** — aperiodic (1/f) fit
- **Blue dotted** — full model (aperiodic + periodic peaks)
- **Red vertical lines** — detected peak centre frequencies

The **slope** printed in the title describes how steeply power decreases with frequency. A steeper slope means more 1/f character.

In [29]:
def plot_fooof_fit(pid, electrode='TP9', save=False):
    """Visualise the FOOOF model fit for one recording."""
    key = 'psd9' if electrode == 'TP9' else 'psd10'
    psd = psd_store[pid][key]

    fm = FOOOF(**FOOOF_SETTINGS)
    fm.fit(FREQS, psd, FOOOF_RANGE)

    # Restrict to the fit range for plotting
    mask      = (FREQS >= FOOOF_RANGE[0]) & (FREQS <= FOOOF_RANGE[1])
    psd_fit   = psd[mask]
    freqs_fit = fm.freqs

    fig, ax = plt.subplots(figsize=(9, 4))
    _add_band_vspans(ax)

    ax.semilogy(FREQS,      psd,              color='#bbb', lw=1.0, zorder=1)
    ax.semilogy(freqs_fit,  psd_fit,          color='#555', lw=1.6,
                label='Raw PSD (fit range)', zorder=2)
    ax.semilogy(freqs_fit,  10**fm._ap_fit,   color='#e07b39', lw=1.8,
                ls='--', label='Aperiodic fit', zorder=4)
    ax.semilogy(freqs_fit,  10**fm.fooofed_spectrum_, color='#2d6a9f',
                lw=1.4, ls=':', label='Full model', zorder=5)

    # Mark detected peaks with vertical lines
    if fm.n_peaks_:
        ymax = psd_fit.max()
        for cf, pw, bw in fm.gaussian_params_:
            ax.axvline(cf, color='#c0392b', lw=0.8, alpha=0.7)
            ax.text(cf + 0.3, ymax * 0.6, f'{cf:.1f}',
                    fontsize=7, color='#c0392b', va='top')

    slope = fm.aperiodic_params_[1]
    ax.set(xlabel='Frequency (Hz)', ylabel='Power (µV²/Hz, log)',
           title=(f'FOOOF — {_short(pid)} | {electrode}'
                  f'\n1/f slope = {slope:.3f}   peaks = {fm.n_peaks_}'),
           xlim=(FREQS[0], FREQS[-1]))
    ax.legend(fontsize=9)
    fig.tight_layout()
    if save:
        fig.savefig(os.path.join(FIGURES_FOLDER, f'{pid}_{electrode}_fooof.png'), dpi=150)
    plt.show()

### Hjorth Parameters Plot

Three grouped bar charts — one per Hjorth parameter — comparing TP9 and TP10 across all recordings.

- **Activity** — signal variance; higher = larger amplitude fluctuations
- **Mobility** — proportional to the mean frequency; higher = faster oscillations
- **Complexity** — how much the spectrum deviates from a pure sine wave; higher = more irregular waveform

In [ ]:
#TODO: Make hjorth activity plot
def plot_hjorth(subset=None, save=False):

### Entropy Measures Plot

Three grouped bar charts — one per entropy type — comparing TP9 and TP10.

| Measure | What it captures |
|---------|------------------|
| **Spectral Entropy** | How evenly power is spread across frequencies (higher = more broadband) |
| **Sample Entropy** | Irregularity of the raw time series (higher = less predictable) |
| **Permutation Entropy** | Ordinal complexity of the signal (higher = more random ordering) |


In [ ]:
#TODO: Make plot to show entropy measures
def plot_entropy(subset=None, save=False):

### Relative power

In [ ]:
#TODO: Make plot to show relative power
def plot_relative_power(subset=None, condition='closed_eyes', save=False):

## 11. Generate All Plots

Now we run all the visualisation functions defined in Section 10.

> **Tip:** Run the cells in order. Plots are saved to `results/figures/` when `save=True`.

In [54]:
# ── Per-participant inspector ──────────────────────────────────────────
# Set the participant number you want to inspect, e.g. 1, 2, 3 ...
INSPECT_PARTICIPANT = 1

pids_for_participant = [
    pid for pid in df_results['participant_id']
    if int(pid.split('_')[0][1:]) == INSPECT_PARTICIPANT
]

if not pids_for_participant:
    print(f'No recordings found for participant {INSPECT_PARTICIPANT}.')
else:
    print(f'Recordings for P{INSPECT_PARTICIPANT:02d}: {pids_for_participant}\n')
    for pid in pids_for_participant:
        plot_mean_psd(pid)
        for elec in ['TP9', 'TP10']:
            plot_spectrogram(pid, electrode=elec)
            plot_fooof_fit(pid, electrode=elec)

### Bar charts: band ratios, symmetry, Hjorth, entropy

Each function produces one figure per timing (pre / post).
Bars are split by group (control vs experimental). Condition: closed eyes.

In [53]:
plot_band_ratios(save=False)
plot_pdBSI(save=False)
plot_hjorth(save=False)
plot_entropy(save=False)
plot_relative_power(save=False)

### Group timeline plot (all four timepoints -- the dot plot)

This is the **main summary visualisation**.
Each panel shows one metric across all four timepoints: S1 pre, S1 post, S2 pre, S2 post.

- **Each dot** = one participant.
- **Short horizontal tick** = group mean.
- **Thin lines** connect the same participant within a session (pre -> post).
- A **dashed vertical line** separates Session 1 from Session 2.

Blue = control group, Orange = experimental group.
Panels are organised by metric family (band ratios, Hjorth, entropy).

In [33]:
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker

GROUP_COLORS = {
    'control':      {'dot': '#378ADD', 'mean': '#185FA5'},
    'experimental': {'dot': '#D85A30', 'mean': '#993C1D'},
}

X_TIMEPOINTS = [(1,'pre'), (1,'post'), (2,'pre'), (2,'post')]
X_LABELS     = ['S1\npre', 'S1\npost', 'S2\npre', 'S2\npost']
# Slight gap between sessions
X_POS        = [0, 1, 2.4, 3.4]
GROUP_OFFSET = {'control': -0.13, 'experimental': 0.13}


def plot_group_metric(metric, condition='closed_eyes', ax=None, save=False):
    """
    Dot plot for one metric across S1pre / S1post / S2pre / S2post.
    Each dot = one participant. Horizontal tick = group mean.
    Control (blue) and Experimental (orange) are offset left/right.
    """
    df_ce = df_results[df_results['condition'] == condition]

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(6, 4))

    rng = np.random.default_rng(42)

    for grp, colors in GROUP_COLORS.items():
        offset = GROUP_OFFSET[grp]
        grp_vals_by_tp = []

        for tp_idx, (sess, tim) in enumerate(X_TIMEPOINTS):
            mask = (
                (df_ce['session'] == sess) &
                (df_ce['timing']  == tim)  &
                (df_ce['group'].str.strip().str.lower() == grp)
            )
            vals = df_ce.loc[mask, metric].dropna().values
            grp_vals_by_tp.append(vals)

            if len(vals) == 0:
                continue

            x_base = X_POS[tp_idx] + offset
            jitter = rng.uniform(-0.06, 0.06, len(vals))

            # Individual dots
            ax.scatter(
                x_base + jitter, vals,
                color=colors['dot'], s=40, alpha=0.75,
                zorder=3, linewidths=0.5, edgecolors='white',
            )

            # Mean tick — short horizontal line
            mean_val = np.mean(vals)
            ax.plot(
                [x_base - 0.10, x_base + 0.10],
                [mean_val, mean_val],
                color=colors['mean'], lw=2.5, zorder=4,
                solid_capstyle='round',
            )

        # Connect same participant across timepoints with thin lines
        # Match participants by their 'participant' column
        participants = df_ce[
            df_ce['group'].str.strip().str.lower() == grp
        ]['participant'].unique()

        for pid in participants:
            for sess in [1, 2]:
                px, py = [], []
                for tp_idx, (s, tim) in enumerate(X_TIMEPOINTS):
                    if s != sess:
                        continue
                    row = df_ce[
                        (df_ce['session']     == sess) &
                        (df_ce['timing']      == tim)  &
                        (df_ce['participant'] == pid)  &
                        (df_ce['group'].str.strip().str.lower() == grp)
                    ]
                    if len(row) == 1 and not np.isnan(row[metric].values[0]):
                        px.append(X_POS[tp_idx] + offset)
                        py.append(row[metric].values[0])
                if len(px) == 2:
                    ax.plot(px, py, color=colors['dot'],
                            lw=0.7, alpha=0.30, zorder=2)

    # Session separator
    mid = (X_POS[1] + X_POS[2]) / 2
    ax.axvline(mid, color='#ccc', lw=0.8, ls='--', zorder=1)

    # Axes formatting
    ax.set_xticks(X_POS)
    ax.set_xticklabels(X_LABELS, fontsize=9)
    ax.set_xlim(X_POS[0] - 0.5, X_POS[-1] + 0.5)
    ax.set_ylabel(metric.replace('_', ' '), fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3g'))

    # Session labels above separator
    ylim = ax.get_ylim()
    ypos = ylim[1] - (ylim[1] - ylim[0]) * 0.04
    ax.text(mid - 0.75, ypos, 'Session 1', ha='center', va='top',
            fontsize=7.5, color='#999')
    ax.text(mid + 0.75, ypos, 'Session 2', ha='center', va='top',
            fontsize=7.5, color='#999')

    if standalone:
        # Legend
        legend_handles = [
            mpatches.Patch(color=GROUP_COLORS['control']['dot'],      label='Control'),
            mpatches.Patch(color=GROUP_COLORS['experimental']['dot'],  label='Experimental'),
        ]
        ax.legend(handles=legend_handles, fontsize=8, frameon=False,
                  loc='upper right')
        ax.set_title(metric.replace('_tp9', ' TP9')
                            .replace('_tp10', ' TP10')
                            .replace('_', ' '), fontsize=10)
        fig.tight_layout()
        if save:
            fname = f'group_{metric}_{condition}.png'
            fig.savefig(os.path.join(FIGURES_FOLDER, fname),
                        dpi=150, bbox_inches='tight')
            print(f'  Saved → {fname}')
        plt.show()


def plot_group_panel(metrics, panel_title, condition='closed_eyes', save=False):
    """Multi-metric panel — one subplot per metric."""
    n     = len(metrics)
    ncols = min(3, n)
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(ncols * 5.0, nrows * 4.0),
        squeeze=False,
    )
    fig.suptitle(f'{panel_title}  ({condition.replace("_", " ")})',
                 fontsize=12, weight='normal', y=1.02)

    for idx, metric in enumerate(metrics):
        ax = axes[idx // ncols][idx % ncols]
        plot_group_metric(metric, condition=condition, ax=ax)
        ax.set_title(
            metric.replace('_tp9', ' TP9')
                  .replace('_tp10', ' TP10')
                  .replace('_', ' '),
            fontsize=10,
        )

    for idx in range(n, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    # Single shared legend at the bottom
    legend_handles = [
        mpatches.Patch(color=GROUP_COLORS['control']['dot'],     label='Control'),
        mpatches.Patch(color=GROUP_COLORS['experimental']['dot'], label='Experimental'),
    ]
    fig.legend(handles=legend_handles, loc='lower center', ncol=2,
               fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.04))

    fig.tight_layout()
    if save:
        fname = f'group_panel_{panel_title.replace(" ","_")}_{condition}.png'
        fig.savefig(os.path.join(FIGURES_FOLDER, fname),
                    dpi=150, bbox_inches='tight')
        print(f'  Saved → {fname}')
    plt.show()

In [34]:
plot_group_panel(['DAR', 'DTABR', 'pdBSI'],
                 panel_title='Band ratios and symmetry', save=True)

plot_group_panel([
    'hjorth_activity_tp9',   'hjorth_activity_tp10',
    'hjorth_mobility_tp9',   'hjorth_mobility_tp10',
    'hjorth_complexity_tp9', 'hjorth_complexity_tp10',
], panel_title='Hjorth parameters', save=True)

plot_group_panel([
    'spectral_entropy_tp9',    'spectral_entropy_tp10',
    'sample_entropy_tp9',      'sample_entropy_tp10',
    'permutation_entropy_tp9', 'permutation_entropy_tp10',
], panel_title='Entropy measures', save=True)

### Alternative view: Boxplot overview

Boxplots show the **distribution** at each timepoint (median, interquartile range, outliers).
This complements the dot plot: instead of individual trajectories you see the group spread.

Use `plot_group_boxplot(metric)` for a single metric or `plot_boxplot_panel(metrics)` for a grid.

In [35]:
import matplotlib.ticker as mticker  # already imported; safe to re-run

def plot_group_boxplot(metric, condition='closed_eyes', ax=None, save=False):
    """Boxplot for one metric across all four timepoints.
    Control (blue) and experimental (orange) shown side by side.
    Jittered individual dots are overlaid for transparency.
    """
    df_ce = df_results[df_results['condition'] == condition]

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(7, 4))

    rng     = np.random.default_rng(42)
    BOX_W   = 0.28
    BOX_SEP = 0.18  # offset between control and experimental

    box_colors = {
        'control':      '#378ADD',
        'experimental': '#D85A30',
    }

    for grp, color in box_colors.items():
        offset = -BOX_SEP if grp == 'control' else BOX_SEP

        for tp_idx, (sess, tim) in enumerate(X_TIMEPOINTS):
            mask = (
                (df_ce['session'] == sess) &
                (df_ce['timing']  == tim)  &
                (df_ce['group'].str.strip().str.lower() == grp)
            )
            vals = df_ce.loc[mask, metric].dropna().values
            if len(vals) == 0:
                continue

            x_pos = X_POS[tp_idx] + offset

            # Compute box stats
            q1, med, q3 = np.percentile(vals, [25, 50, 75])
            iqr = q3 - q1
            lo_w = max(vals.min(), q1 - 1.5 * iqr)
            hi_w = min(vals.max(), q3 + 1.5 * iqr)
            outliers = vals[(vals < lo_w) | (vals > hi_w)]

            # IQR box
            ax.bar(x_pos, q3 - q1, BOX_W, bottom=q1,
                   color=color, alpha=0.22, edgecolor=color, linewidth=1.2, zorder=2)
            # Median line
            ax.plot([x_pos - BOX_W/2, x_pos + BOX_W/2], [med, med],
                    color=color, lw=2.2, zorder=3)
            # Whiskers
            ax.plot([x_pos, x_pos], [lo_w, q1], color=color, lw=0.9, zorder=2)
            ax.plot([x_pos, x_pos], [q3, hi_w], color=color, lw=0.9, zorder=2)
            ax.plot([x_pos - BOX_W/4, x_pos + BOX_W/4], [lo_w, lo_w],
                    color=color, lw=0.9, zorder=2)
            ax.plot([x_pos - BOX_W/4, x_pos + BOX_W/4], [hi_w, hi_w],
                    color=color, lw=0.9, zorder=2)
            # Outlier dots
            if len(outliers):
                ax.scatter([x_pos] * len(outliers), outliers,
                           color=color, s=22, zorder=5,
                           edgecolors='white', linewidths=0.5)
            # Jittered individual dots
            jitter = rng.uniform(-BOX_W * 0.28, BOX_W * 0.28, len(vals))
            ax.scatter(x_pos + jitter, vals,
                       color=color, s=16, alpha=0.55, zorder=4, edgecolors='none')

    mid = (X_POS[1] + X_POS[2]) / 2
    ax.axvline(mid, color='#ccc', lw=0.8, ls='--', zorder=1)
    ax.set_xticks(X_POS)
    ax.set_xticklabels(X_LABELS, fontsize=9)
    ax.set_xlim(X_POS[0] - 0.55, X_POS[-1] + 0.55)
    ax.set_ylabel(metric.replace('_', ' '), fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3g'))

    ylim = ax.get_ylim()
    ypos = ylim[1] - (ylim[1] - ylim[0]) * 0.04
    ax.text(mid - 0.75, ypos, 'Session 1', ha='center', va='top', fontsize=7.5, color='#999')
    ax.text(mid + 0.75, ypos, 'Session 2', ha='center', va='top', fontsize=7.5, color='#999')

    if standalone:
        legend_handles = [
            mpatches.Patch(color=box_colors['control'],      label='Control'),
            mpatches.Patch(color=box_colors['experimental'], label='Experimental'),
        ]
        ax.legend(handles=legend_handles, fontsize=8, frameon=False, loc='upper right')
        ax.set_title(
            metric.replace('_tp9', ' TP9')
                  .replace('_tp10', ' TP10')
                  .replace('_', ' '), fontsize=10)
        fig.tight_layout()
        if save:
            fname = f'boxplot_{metric}_{condition}.png'
            fig.savefig(os.path.join(FIGURES_FOLDER, fname), dpi=150, bbox_inches='tight')
            print(f'  Saved -> {fname}')
        plt.show()


def plot_boxplot_panel(metrics, panel_title, condition='closed_eyes', save=False):
    """Multi-metric boxplot panel -- one subplot per metric."""
    n     = len(metrics)
    ncols = min(3, n)
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(ncols * 5.5, nrows * 4.0),
        squeeze=False,
    )
    fig.suptitle(f'{panel_title}  ({condition.replace("_", " ")})  -- boxplots',
                 fontsize=12, weight='normal', y=1.02)

    for idx, metric in enumerate(metrics):
        ax = axes[idx // ncols][idx % ncols]
        plot_group_boxplot(metric, condition=condition, ax=ax)
        ax.set_title(
            metric.replace('_tp9', ' TP9')
                  .replace('_tp10', ' TP10')
                  .replace('_', ' '), fontsize=10)

    for idx in range(n, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    legend_handles = [
        mpatches.Patch(color=GROUP_COLORS['control']['dot'],      label='Control'),
        mpatches.Patch(color=GROUP_COLORS['experimental']['dot'], label='Experimental'),
    ]
    fig.legend(handles=legend_handles, loc='lower center', ncol=2,
               fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.04))
    fig.tight_layout()
    if save:
        fname = f'boxplot_panel_{panel_title.replace(" ", "_")}_{condition}.png'
        fig.savefig(os.path.join(FIGURES_FOLDER, fname), dpi=150, bbox_inches='tight')
        print(f'  Saved -> {fname}')
    plt.show()


In [36]:
plot_boxplot_panel(['DAR', 'DTABR', 'pdBSI'],
                  panel_title='Band ratios and symmetry', save=True)

plot_boxplot_panel([
    'hjorth_activity_tp9',   'hjorth_activity_tp10',
    'hjorth_mobility_tp9',   'hjorth_mobility_tp10',
    'hjorth_complexity_tp9', 'hjorth_complexity_tp10',
], panel_title='Hjorth parameters', save=True)

plot_boxplot_panel([
    'spectral_entropy_tp9',    'spectral_entropy_tp10',
    'sample_entropy_tp9',      'sample_entropy_tp10',
    'permutation_entropy_tp9', 'permutation_entropy_tp10',
], panel_title='Entropy measures', save=True)

## 12. Statistical Analysis

This section tests whether the EEG metrics change significantly across **groups, sessions, and timepoints**.

### Design

| Factor | Type | Levels |
|--------|------|--------|
| **Group** | Between-subjects | control / experimental |
| **Session** | Within-subjects | 1 / 2 |
| **Timing** | Within-subjects | pre / post |

We use a **2 x 2 x 2 mixed ANOVA** (pingouin library).
The four within-subject timepoints are: `S1_pre`, `S1_post`, `S2_pre`, `S2_post`.

**Multiple comparison correction:** Benjamini-Hochberg FDR, applied separately per ANOVA source term.

**Non-parametric fallback:** Metrics that violate normality (Shapiro-Wilk) are also tested with Friedman + Mann-Whitney.

> **Requires:** `participants.csv` with columns: participant ID, group, sex, handedness, date of birth.
> **Condition used:** closed eyes only.